# Create HSC R&D Division Awards (Public Health Agency, Northern Ireland)

Creates awards made by the **HSC Research & Development Division of the Public
Health Agency (Northern Ireland)** from the award-portfolio tables the division
publishes at https://research.hscni.net (one Drupal page per scheme: Fellowship
Awards, Enabling Research Awards, Opportunity-Led Commissioned Research,
Recognised Research Group awards, US-Ireland R&D Partnership, PPI small grants,
COVID-19 Rapid Response, GPARTS, Cochrane Fellowships, NILS awards). Each row
gives title, chief investigator / fellow (with institution abbreviation, e.g.
`QUB/BHSCT`), dates, status and HSC R&D cost in GBP. **625 awards, 1998-2026,
~99% PI and amount, GBP 103M total** (local run 2026-09-30).

**Prerequisites:**
- Run `scripts/local/hscni_pha_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/hscni_pha/hscni_pha_projects.parquet` (§1.4 shrink guard).

**Funder choice.** The HSC R&D Division is part of the Public Health Agency
(F4320320853, ROR 03ek62e72, Crossref Funder 10.13039/501100001626, Belfast).
OpenAlex also has a separate, ROR-less funder row "Health and Social Care
Research and Development Division" (F4320335543). Citing works overwhelmingly
attribute HSC R&D grants to the PHA row (e.g. `COM/5596/20`, `EAT/4735/12`
under F4320320853: ~370 acknowledgement rows vs ~60 under F4320335543), so
these awards are assigned to the PHA. F4320335543 is a candidate funder merge.

**`funder_award_id` (§2.1.1):** the citation side uses HSC R&D references
(`COM/nnnn/yy`, `EAT/nnnn/yy`, `STL/…`, `CHI/…`, `RES/…`), but the portfolio
pages do not print them, so they cannot be joined. The script ships a stable
synthetic id `HSCRD-{page code}-{sha1(page|title|PI family)[:10]}` (page codes
FEL, ERA, OPP, RRG, USI, PPI, COV, GPA, COC, NILS). These awards therefore do
not collapse onto the existing acknowledgement shells.

**Amounts** are the HSC R&D contribution in GBP ("HSC R&D cost"; for co-funded
schemes the co-funders are named in `co_funders`, not included in the amount).
Dates are mostly years only; `start_date` / `end_date` are Jan 1 / Dec 31 of the
published years unless the page gives exact dates (PPI grants).

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320320853`,
metadata read from `openalex.funders.funders`.

**Priority:** `501` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hscni_pha_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hscni_pha/hscni_pha_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.hscni_pha_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.hscni_pha_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hscni_pha_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320853 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320853;

## Step 2: Create Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hscni_pha_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320853  -- Public Health Agency (Northern Ireland)
),

g AS (
    SELECT
        r.*,
        NULLIF(TRIM(r.lead_given_name), '') AS given,
        NULLIF(TRIM(r.lead_family_name), '') AS family,
        -- "N/A" / "TBC" titles (Pre-Doctoral Bridging, ICAT placements)
        CASE WHEN UPPER(TRIM(r.title)) IN ('N/A', 'TBC', 'TBA', '') THEN NULL ELSE TRIM(r.title) END AS clean_title,
        TRY_CAST(r.start_year AS INT) AS sy,
        TRY_CAST(r.end_year AS INT) AS ey
    FROM openalex.awards.hscni_pha_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(g.clean_title,
             CONCAT_WS(' - ', NULLIF(TRIM(g.scheme), ''), CONCAT_WS(' ', g.given, g.family))) as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'GBP' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.page_code IN ('FEL', 'COC', 'GPA') OR LOWER(g.scheme) LIKE '%fellowship%' THEN 'fellowship'
         ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.scheme), '') as funder_scheme,
    'hscni_rd_portfolio' as provenance,
    COALESCE(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd'), make_date(g.sy, 1, 1)) as start_date,
    COALESCE(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd'), make_date(g.ey, 12, 31)) as end_date,
    COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), g.sy) as start_year,
    COALESCE(YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')), g.ey) as end_year,
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.pi_affiliation), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): every shipped id is the synthetic HSCRD form, and no id repeats.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^HSCRD-(FEL|ERA|OPP|RRG|USI|PPI|COV|GPA|COC|NILS)-[0-9a-f]{10}$' THEN 0 ELSE 1 END) AS bad_shape
FROM openalex.awards.hscni_pha_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hscni_rd_portfolio' AND priority = 501;

-- Insert into openalex_awards_raw with priority.
-- Priority 501: direct-from-funder ingest of the HSC R&D Division award portfolios.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    501 as priority
FROM openalex.awards.hscni_pha_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.hscni_pha_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_gbp
FROM openalex.awards.hscni_pha_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_gbp
FROM openalex.awards.hscni_pha_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.hscni_pha_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect ~100% title, ~99% amount and PI, ~99% start year).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.hscni_pha_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hscni_rd_portfolio'
GROUP BY provenance, priority;